In [1]:
import pandas as pd
import numpy as np
from itertools import product

# 1. Load the raw data (navigating up from the src/ folder)
sales = pd.read_csv('../data/sales_train.csv')
items = pd.read_csv('../data/items.csv')
shops = pd.read_csv('../data/shops.csv')
test = pd.read_csv('../data/test.csv')

# 2. Clean Extreme Outliers
sales = sales[sales['item_price'] < 100000]
sales = sales[sales['item_cnt_day'] < 1000]
sales = sales[sales['item_price'] > 0]

# 3. Aggregate to Monthly Sales
monthly_sales = sales.groupby(['date_block_num', 'shop_id', 'item_id']).agg({
    'item_cnt_day': 'sum',
    'item_price': 'mean' 
}).reset_index()

monthly_sales.rename(columns={'item_cnt_day': 'item_cnt_month'}, inplace=True)
monthly_sales['item_cnt_month'] = monthly_sales['item_cnt_month'].clip(0, 20)

In [2]:
# 1. Create a grid of all active (shop_id, item_id) combinations per month
matrix = []
cols = ['date_block_num', 'shop_id', 'item_id']

for i in range(34): # date_block_num goes from 0 to 33
    sales_month = sales[sales['date_block_num'] == i]
    matrix.append(np.array(list(product(
        [i], 
        sales_month['shop_id'].unique(), 
        sales_month['item_id'].unique()
    )), dtype='int16'))

# Convert the list of arrays into a DataFrame
matrix = pd.DataFrame(np.vstack(matrix), columns=cols)

# Downcast memory types to prevent RAM crashes
matrix['date_block_num'] = matrix['date_block_num'].astype(np.int8)
matrix['shop_id'] = matrix['shop_id'].astype(np.int8)
matrix['item_id'] = matrix['item_id'].astype(np.int16)
matrix.sort_values(cols, inplace=True)

# 2. Merge the aggregated monthly sales onto this master grid
train_data = pd.merge(matrix, monthly_sales, on=cols, how='left')

# 3. Fill missing values (combinations that didn't sell get a 0)
train_data['item_cnt_month'] = train_data['item_cnt_month'].fillna(0).astype(np.float16)
train_data['item_price'] = train_data['item_price'].fillna(0).astype(np.float16)

print(train_data.head())
print(f"Grid size: {len(train_data)} rows")

   date_block_num  shop_id  item_id  item_cnt_month  item_price
0               0        0       19             0.0         0.0
1               0        0       27             0.0         0.0
2               0        0       28             0.0         0.0
3               0        0       29             0.0         0.0
4               0        0       32             6.0       221.0
Grid size: 10913804 rows


c:\Users\Pradeep\anaconda3\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# 1. Append the Test Set
test['date_block_num'] = 34
test['date_block_num'] = test['date_block_num'].astype(np.int8)
test['shop_id'] = test['shop_id'].astype(np.int8)
test['item_id'] = test['item_id'].astype(np.int16)

train_data = pd.concat([train_data, test.drop(['ID'], axis=1)], ignore_index=True, sort=False, keys=cols)
train_data.fillna(0, inplace=True) 

# 2. Define the Lag Feature Function
def create_lag_features(df, lags, col):
    tmp = df[['date_block_num', 'shop_id', 'item_id', col]]
    for i in lags:
        shifted = tmp.copy()
        shifted.columns = ['date_block_num', 'shop_id', 'item_id', col + f'_lag_{i}']
        shifted['date_block_num'] += i
        df = pd.merge(df, shifted, on=['date_block_num', 'shop_id', 'item_id'], how='left')
    return df

# 3. Apply the Lag Feature Function (1, 2, 3, 6, and 12 months back)
train_data = create_lag_features(train_data, [1, 2, 3, 6, 12], 'item_cnt_month')

for lag in [1, 2, 3, 6, 12]:
    train_data[f'item_cnt_month_lag_{lag}'] = train_data[f'item_cnt_month_lag_{lag}'].fillna(0).astype(np.float16)

# 4. Drop the first 11 months (they do not have 12 months of history)
train_data = train_data[train_data['date_block_num'] >= 12]

C:\Users\Pradeep\AppData\Local\Temp\ipykernel_44920\2607489927.py:7: FutureWarning: The behavior of pd.concat with len(keys) != len(objs) is deprecated. In a future version this will raise instead of truncating to the smaller of the two sequences
  train_data = pd.concat([train_data, test.drop(['ID'], axis=1)], ignore_index=True, sort=False, keys=cols)


Memory Optimization: Data science projects often fail because they exhaust system RAM during the feature engineering phase. By forcing integer downcasting (int8, int16) and filling missing values aggressively with float16, this code ensures the 11-million-row dataframe will run smoothly on your local machine.

Time Leakage Prevention: By calculating lags based explicitly on date_block_num rather than just shifting the dataframe index, this function guarantees that month 34 looks exclusively at historical months and never peeks at future data.

Because calculating the 12-month lag requires a full year of history, the first 11 months of your dataset (months 0 through 11) will now have incomplete lag features. You must drop them from the training set.

Add train_data = train_data[train_data['date_block_num'] >= 12] to remove those incomplete rows.